In [0]:
# --- SETUP ---
csv_path = "/Volumes/workspace/default/volume/dashboard_data.csv"
table_name = "workspace.default.dashboard_data"

# Sicherheits-Check: Existiert die CSV Datei?
try:
    dbutils.fs.ls(csv_path)
    print(f"✅ Datei gefunden: {csv_path}")
except:
    print(f"⚠️ Datei fehlt. Erstelle Dummy-CSV für die Demo...")
    
    # Wir erstellen kurz einen DataFrame und speichern ihn als CSV
    # Damit dein eigentlicher Code gleich funktioniert
    data = [
        ("101", "Data Engineering", "High"), 
        ("102", "Data Science", "Medium"),
        ("103", "AI Basics", "Low")
    ]
    (spark.createDataFrame(data, ["id", "topic", "priority"])
     .write.mode("overwrite").option("header", "true").csv(csv_path))
    
    print("✅ Dummy-CSV erstellt.")

In [0]:
# --- INGESTION ---
print(f"🚀 Lese CSV von {csv_path}...")

df = spark.read.csv(
    csv_path,
    header=True,
    inferSchema=True
)

print(f"💾 Speichere in Tabelle {table_name}...")

# Tabelle zuerst löschen, falls sie existiert
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

# Jetzt neu anlegen
(df.write
   .format("delta")
   .saveAsTable(table_name)
)

print("✅ Tabelle erfolgreich erstellt.")

In [0]:
%sql
-- Jetzt können wir ganz normales SQL machen
SELECT * FROM workspace.default.dashboard_data

In [0]:
# Genie abfrage starten

# Cleanup

In [0]:
# --- CLEANUP ---
# Löscht die Tabelle wieder aus dem Metastore
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

# Optional: Willst du auch die CSV-Quelldatei löschen?
# dbutils.fs.rm(csv_path)

print(f"🗑️ Tabelle {table_name} gelöscht.")